# R&D-7. Эмбеддинги как adjustment set — ноутбук для внутреннего контура

**Самодостаточный ноутбук.** Causal-слой пакета `rnd_reports.embeddings` (numpy/sklearn, без pyspark) вшит ниже в ячейки — импортов `rnd_reports` нет, файл переносится во внутренний контур одним куском (нужны `numpy / pandas / scikit-learn`).

**Как использовать во внутреннем контуре:**
1. В ячейке-конфиге укажи `INPUT_CSV` — путь к своей таблице (контракт `epk_id, report_dt, col_*, treatment, outcome`).
2. Ячейку-генератор синтетики (помечена ⚠️) **не запускай** — она только для проверки запускаемости в репозитории.
3. Прогони остальные ячейки: снижение эмбеддингов (PCA) → propensity → оценка ATE с поправкой (naive / IPW / doubly-robust) + диагностика баланса и overlap.

Идея R&D-7: в наблюдательных данных назначение трита коррелирует с признаками (селекция) → наивная разность средних смещена; если сокращённые эмбеддинги покрывают конфаундеры, поправка на них убирает смещение. `true_ate` известен только для синтетики.

In [1]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

## Библиотека (вшита из `rnd_reports.embeddings`)

In [2]:
# === Вшито из rnd_reports.embeddings/contracts.py ===
"""Контракты входных/выходных схем адаптеров R&D-7 (без Spark-зависимостей).

Step 1: только проверки структуры и имена колонок — **без алгоритмов**. Сами
адаптеры (PCA-снижение, propensity) живут в :mod:`reducer` / :mod:`propensity`.

Хелперы принимают любой объект с атрибутом ``.columns`` (в т.ч. pyspark
``DataFrame``), поэтому pyspark здесь не импортируется.
"""


# --- имена ключевых колонок и префиксов ---
EPK_ID = "epk_id"
REPORT_DT = "report_dt"
TREATMENT = "treatment"
KEY_COLUMNS = (EPK_ID, REPORT_DT)

EMBEDDING_PREFIX = "col_"  # входные эмбеддинги: col_000, col_001, ...
REDUCED_PREFIX = "emb_"  # выход reducer'а: emb_000, emb_001, ...
PROPENSITY_SCORE = "propensity_score"  # выход propensity-адаптера


def _embedding_sort_key(name: str) -> tuple[int, object]:
    """Натуральная сортировка ``col_*``: числовой суффикс — как число, иначе строка."""
    suffix = name[len(EMBEDDING_PREFIX) :]
    return (0, int(suffix)) if suffix.isdigit() else (1, suffix)


def embedding_feature_columns(df) -> list[str]:
    """Эмбеддинг-колонки (префикс ``col_``) в естественном порядке индексов."""
    cols = [c for c in df.columns if c.startswith(EMBEDDING_PREFIX)]
    return sorted(cols, key=_embedding_sort_key)


def reduced_column_names(reducted_shape: int) -> list[str]:
    """Имена выходных колонок reducer'а: ``emb_000 ... emb_{reducted_shape-1}``."""
    if reducted_shape < 1:
        raise ValueError(f"reducted_shape должен быть >= 1, получено {reducted_shape}")
    return [f"{REDUCED_PREFIX}{i:03d}" for i in range(reducted_shape)]


def validate_embedding_schema(df) -> list[str]:
    """Проверить схему эмбеддинг-датасета; вернуть список эмбеддинг-колонок.

    Требуется ``epk_id``, ``report_dt`` и >=1 колонка с префиксом ``col_``.
    """
    cols = set(df.columns)
    missing = [c for c in KEY_COLUMNS if c not in cols]
    if missing:
        raise ValueError(f"В эмбеддинг-датасете нет ключевых колонок: {missing}")
    embeddings = embedding_feature_columns(df)
    if not embeddings:
        raise ValueError(
            f"Не найдено ни одной эмбеддинг-колонки с префиксом {EMBEDDING_PREFIX!r}"
        )
    return embeddings


def validate_treatment_schema(df) -> None:
    """Проверить схему датасета трита: нужны ``epk_id``, ``report_dt``, ``treatment``."""
    cols = set(df.columns)
    missing = [c for c in (*KEY_COLUMNS, TREATMENT) if c not in cols]
    if missing:
        raise ValueError(f"В датасете трита нет колонок: {missing}")


In [3]:
# === Вшито из rnd_reports.embeddings/synthetic.py ===
"""Синтетический наблюдательный сценарий R&D-7 (numpy/pandas, без pyspark).

Генерирует данные со схемой эмбеддингов (``epk_id, report_dt, col_*``) + ``treatment`` и
``outcome`` с **известным** ATE. Назначение трита НЕ рандомизировано: зависит от латентных
конфаундеров через эмбеддинги (селекция). Эмбеддинги покрывают конфаундеры, поэтому поправка
на них должна убирать смещение наивной оценки.

Используется ноутбуком rnd/07 и тестами; ground-truth ATE известен по построению.
"""


from dataclasses import dataclass, field

import numpy as np
import pandas as pd


@dataclass
class EmbeddingScenario:
    data: pd.DataFrame              # epk_id, report_dt, col_000.., treatment, outcome
    true_ate: float
    embedding_columns: list[str]
    params: dict = field(default_factory=dict)


def make_embedding_observational_scenario(
    n: int = 4000,
    k: int = 8,
    *,
    n_latent: int = 3,
    true_ate: float = 3.0,
    n_months: int = 6,
    confounding: float = 1.5,
    noise_sd: float = 1.0,
    seed: int = 42,
) -> EmbeddingScenario:
    """Сгенерировать наблюдательный сценарий «эмбеддинги как adjustment set».

    - латентные конфаундеры ``z`` (n × n_latent) ~ N(0, 1);
    - эмбеддинги ``col_0..col_{k-1}`` = ``z @ W + шум`` (несут информацию о конфаундерах);
    - ``treatment ~ Bernoulli(sigmoid(confounding · z·a))`` — селекция, не рандом;
    - ``outcome = true_ate·T + z·b + ε`` (конфаундеры влияют и на трит, и на исход);
    - ``report_dt`` — равномерно по ``n_months`` месячным срезам (для in-time демо).
    """
    rng = np.random.default_rng(seed)
    z = rng.standard_normal((n, n_latent))

    # Эмбеддинги: зашумлённая линейная проекция конфаундеров (k >= n_latent желательно).
    w = rng.standard_normal((n_latent, k))
    emb = z @ w + rng.standard_normal((n, k)) * 0.5

    # Назначение трита: логистическая селекция по конфаундерам.
    a = rng.standard_normal(n_latent)
    logits = confounding * (z @ a)
    p_treat = 1.0 / (1.0 + np.exp(-logits))
    treatment = (rng.uniform(size=n) < p_treat).astype(int)

    # Исход: эффект трита + влияние конфаундеров + шум.
    b = rng.standard_normal(n_latent)
    outcome = true_ate * treatment + z @ b + rng.standard_normal(n) * noise_sd

    emb_cols = [f"col_{i:03d}" for i in range(k)]
    df = pd.DataFrame(emb, columns=emb_cols)
    df.insert(0, "epk_id", np.arange(n))
    # Месячные срезы report_dt (строки YYYY-MM-01).
    months = pd.date_range("2024-01-01", periods=n_months, freq="MS")
    df["report_dt"] = months[rng.integers(0, n_months, size=n)].strftime("%Y-%m-%d")
    df["treatment"] = treatment
    df["outcome"] = outcome

    return EmbeddingScenario(
        data=df,
        true_ate=float(true_ate),
        embedding_columns=emb_cols,
        params=dict(n=n, k=k, n_latent=n_latent, n_months=n_months,
                    confounding=confounding, noise_sd=noise_sd, seed=seed),
    )


In [4]:
# === Вшито из rnd_reports.embeddings/experiment.py ===
"""R&D-7: эмбеддинги как adjustment set в наблюдательном (нерандомизированном) испытании.

Реализация causal-слоя поверх сокращённых эмбеддингов. Всё на ``numpy``/``scikit-learn``
(без pyspark) — запускается в base-окружении и переиспользуется ноутбуком/тестами.

Идея: в наблюдательных данных назначение трита коррелирует с признаками (селекция), поэтому
наивная разность средних смещена. Если сокращённые эмбеддинги покрывают конфаундеры, то
поправка на них (propensity-взвешивание / doubly-robust) убирает смещение. Здесь — оценка ATE
с поправкой, диагностика баланса и overlap, и сравнение с эталонным эффектом.
"""


from typing import Any, Mapping, Optional

import numpy as np

_EPS = 1e-6


def _as_2d(x: Any) -> np.ndarray:
    arr = np.asarray(x, dtype=float)
    return arr.reshape(-1, 1) if arr.ndim == 1 else arr


def _as_1d(x: Any) -> np.ndarray:
    return np.asarray(x, dtype=float).ravel()


def fit_propensity(reduced_embeddings: Any, treatment: Any, *,
                   max_iter: int = 1000, C: float = 1.0) -> np.ndarray:
    """Propensity ``P(T=1 | reduced_embeddings)`` через ``LogisticRegression``.

    Возвращает вектор вероятностей, клипнутый в ``[_EPS, 1-_EPS]`` (для устойчивости IPW).
    """
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler

    x = _as_2d(reduced_embeddings)
    t = _as_1d(treatment).astype(int)
    xs = StandardScaler().fit_transform(x)
    model = LogisticRegression(max_iter=max_iter, C=C)
    model.fit(xs, t)
    p = model.predict_proba(xs)[:, 1]
    return np.clip(p, _EPS, 1.0 - _EPS)


def _ipw_ate(t: np.ndarray, y: np.ndarray, p: np.ndarray) -> float:
    """Стабилизированный IPW-ATE (Hajek): нормировка весов внутри каждой группы."""
    w1 = t / p
    w0 = (1.0 - t) / (1.0 - p)
    mu1 = np.sum(w1 * y) / np.sum(w1)
    mu0 = np.sum(w0 * y) / np.sum(w0)
    return float(mu1 - mu0)


def _dr_ate(x: np.ndarray, t: np.ndarray, y: np.ndarray, p: np.ndarray) -> float:
    """Doubly-robust (AIPW): per-arm линейная регрессия исхода + IPW-коррекция остатков."""
    from sklearn.linear_model import LinearRegression

    mu1_model = LinearRegression().fit(x[t == 1], y[t == 1])
    mu0_model = LinearRegression().fit(x[t == 0], y[t == 0])
    mu1 = mu1_model.predict(x)
    mu0 = mu0_model.predict(x)
    psi = (mu1 - mu0
           + t * (y - mu1) / p
           - (1.0 - t) * (y - mu0) / (1.0 - p))
    return float(np.mean(psi))


def estimate_ate_with_adjustment(
    reduced_embeddings: Any,
    treatment: Any,
    outcome: Any,
    *,
    method: str = "propensity_weighting",
) -> float:
    """Оценить ATE в наблюдательных данных с поправкой на эмбеддинги.

    ``method``:
    - ``"naive"`` — разность средних без поправки (смещена при селекции);
    - ``"propensity_weighting"`` — стабилизированный IPW по propensity на эмбеддингах;
    - ``"doubly_robust"`` — AIPW (per-arm регрессия исхода + IPW-коррекция).
    """
    t = _as_1d(treatment)
    y = _as_1d(outcome)
    if method == "naive":
        return float(y[t == 1].mean() - y[t == 0].mean())
    x = _as_2d(reduced_embeddings)
    p = fit_propensity(x, t)
    if method == "propensity_weighting":
        return _ipw_ate(t, y, p)
    if method == "doubly_robust":
        return _dr_ate(x, t, y, p)
    raise ValueError(
        f"Неизвестный method={method!r}; ожидается naive/propensity_weighting/doubly_robust"
    )


def _smd(values: np.ndarray, t: np.ndarray, w: Optional[np.ndarray] = None) -> float:
    """Standardized mean difference одной ковариаты (опц. с весами)."""
    if w is None:
        m1, m0 = values[t == 1].mean(), values[t == 0].mean()
        v1, v0 = values[t == 1].var(), values[t == 0].var()
    else:
        def wm(mask):
            ww = w[mask]
            return np.sum(ww * values[mask]) / np.sum(ww)

        def wv(mask, mu):
            ww = w[mask]
            return np.sum(ww * (values[mask] - mu) ** 2) / np.sum(ww)

        m1, m0 = wm(t == 1), wm(t == 0)
        v1, v0 = wv(t == 1, m1), wv(t == 0, m0)
    pooled = np.sqrt((v1 + v0) / 2.0)
    return float(abs(m1 - m0) / pooled) if pooled > _EPS else 0.0


def covariate_balance_after_adjustment(
    reduced_embeddings: Any,
    treatment: Any,
    *,
    propensity_score: Any | None = None,
) -> Mapping[str, float]:
    """Баланс ковариат (|SMD|) по эмбеддингам до и после IPW-взвешивания.

    Возвращает сводку: max/mean ``|SMD|`` до и после. Хороший adjustment set уменьшает
    дисбаланс после взвешивания.
    """
    x = _as_2d(reduced_embeddings)
    t = _as_1d(treatment)
    p = (_as_1d(propensity_score) if propensity_score is not None
         else fit_propensity(x, t))
    w = t / p + (1.0 - t) / (1.0 - p)
    before = [_smd(x[:, j], t) for j in range(x.shape[1])]
    after = [_smd(x[:, j], t, w) for j in range(x.shape[1])]
    return {
        "n_covariates": float(x.shape[1]),
        "max_abs_smd_before": float(np.max(before)),
        "mean_abs_smd_before": float(np.mean(before)),
        "max_abs_smd_after": float(np.max(after)),
        "mean_abs_smd_after": float(np.mean(after)),
    }


def overlap_diagnostics(propensity_score: Any) -> Mapping[str, float]:
    """Overlap/positivity по распределению propensity score.

    Хороший overlap: масса в середине [0.1, 0.9], мало экстремальных значений.
    """
    p = _as_1d(propensity_score)
    return {
        "min": float(p.min()),
        "max": float(p.max()),
        "p01": float(np.percentile(p, 1)),
        "p99": float(np.percentile(p, 99)),
        "frac_in_0.1_0.9": float(np.mean((p >= 0.1) & (p <= 0.9))),
        "frac_below_0.05": float(np.mean(p < 0.05)),
        "frac_above_0.95": float(np.mean(p > 0.95)),
    }


def evaluate_adjustment_set_quality(
    reduced_embeddings: Any,
    treatment: Any,
    outcome: Any,
    *,
    ground_truth_effect: float | None = None,
) -> Mapping[str, float]:
    """Качество adjustment set из сокращённых эмбеддингов.

    Считает ATE тремя способами (naive / IPW / DR), баланс и overlap. При известном
    ``ground_truth_effect`` добавляет смещения и относительное снижение смещения IPW vs naive.
    """
    x = _as_2d(reduced_embeddings)
    t = _as_1d(treatment)
    y = _as_1d(outcome)
    p = fit_propensity(x, t)

    ate_naive = estimate_ate_with_adjustment(x, t, y, method="naive")
    ate_ipw = _ipw_ate(t, y, p)
    ate_dr = _dr_ate(x, t, y, p)

    out: dict[str, float] = {
        "ate_naive": ate_naive,
        "ate_ipw": ate_ipw,
        "ate_dr": ate_dr,
    }
    bal = covariate_balance_after_adjustment(x, t, propensity_score=p)
    out["max_abs_smd_before"] = bal["max_abs_smd_before"]
    out["max_abs_smd_after"] = bal["max_abs_smd_after"]
    ov = overlap_diagnostics(p)
    out["overlap_frac_in_0.1_0.9"] = ov["frac_in_0.1_0.9"]

    if ground_truth_effect is not None:
        g = float(ground_truth_effect)
        out["ground_truth"] = g
        out["bias_naive"] = ate_naive - g
        out["bias_ipw"] = ate_ipw - g
        out["bias_dr"] = ate_dr - g
        denom = abs(ate_naive - g)
        if denom > _EPS:
            out["abs_bias_reduction_ipw_pct"] = float(
                (1.0 - abs(ate_ipw - g) / denom) * 100.0
            )
            out["abs_bias_reduction_dr_pct"] = float(
                (1.0 - abs(ate_dr - g) / denom) * 100.0
            )
    return out


## 1. Данные: одна таблица из `INPUT_CSV`

Контракт таблицы: `epk_id`, `report_dt`, эмбеддинги `col_000, col_001, ...`, `treatment` (0/1), `outcome`. Генератор пишет одну демонстрационную таблицу в `INPUT_CSV`; во внутреннем контуре его пропускают и подставляют реальную таблицу.

In [5]:
# --- Конфигурация прогона -------------------------------------------------------
INPUT_CSV = "data/07_embedding_adjustment_set/sample.csv"  # во внутреннем контуре — своя таблица
N_COMPONENTS = 5     # размер adjustment set после PCA-снижения эмбеддингов
SEED = 11
TRUE_ATE = None      # реальные данные: истинный ATE неизвестен

pd.set_option('display.width', 200)
pd.set_option('display.precision', 4)

In [6]:
# +++ ЯЧЕЙКА-ГЕНЕРАТОР СИНТЕТИКИ +++
# +++ НЕ ЗАПУСКАТЬ ВО ВНУТРЕННЕМ КОНТУРЕ +++
# Синтез одной демонстрационной таблицы со схемой эмбеддингов -> запись в INPUT_CSV.
# Нужна только для проверки запускаемости в репозитории.
from pathlib import Path

sc = make_embedding_observational_scenario(n=4000, k=8, true_ate=3.0, confounding=1.5, seed=SEED)
Path(INPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
sc.data.to_csv(INPUT_CSV, index=False)
TRUE_ATE = sc.true_ate
print(f"синтетическая таблица записана в {INPUT_CSV}: {len(sc.data)} юнитов, истинный ATE = {TRUE_ATE}")

синтетическая таблица записана в data/07_embedding_adjustment_set/sample.csv: 4000 юнитов, истинный ATE = 3.0


In [7]:
# Загрузка таблицы. Во внутреннем контуре сюда приходит реальная таблица из INPUT_CSV.
df = pd.read_csv(INPUT_CSV)
emb_cols = embedding_feature_columns(df)
emb = df[emb_cols].to_numpy()
t = df["treatment"].to_numpy()
y = df["outcome"].to_numpy()
print(f"таблица: {len(df)} юнитов, эмбеддинг-колонок: {len(emb_cols)}, доля трита: {t.mean():.3f}")

# Снижение размерности эмбеддингов до adjustment set + propensity.
reduced = PCA(n_components=N_COMPONENTS, random_state=0).fit_transform(emb)
p = fit_propensity(reduced, t)
print(f"reduced shape: {reduced.shape} | propensity: min={p.min():.3f} max={p.max():.3f}")

таблица: 4000 юнитов, эмбеддинг-колонок: 8, доля трита: 0.502
reduced shape: (4000, 5) | propensity: min=0.018 max=0.974


## 2. Оценка ATE с поправкой на эмбеддинги (single-table)

Сравниваем наивную разность средних, IPW (взвешивание по propensity) и doubly-robust на одной таблице; при известном `TRUE_ATE` добавляется смещение. Диагностика баланса (|SMD| до/после) и overlap показывает, покрывают ли эмбеддинги конфаундеры.

In [8]:
res = {
    "naive": estimate_ate_with_adjustment(reduced, t, y, method="naive"),
    "IPW (propensity)": estimate_ate_with_adjustment(reduced, t, y, method="propensity_weighting"),
    "doubly_robust": estimate_ate_with_adjustment(reduced, t, y, method="doubly_robust"),
}
tbl = pd.DataFrame({"ATE": res})
if TRUE_ATE is not None:
    tbl["bias_vs_true"] = tbl["ATE"] - TRUE_ATE
print("=== Оценки ATE по методам ===")
display(tbl.round(3))

bal = covariate_balance_after_adjustment(reduced, t, propensity_score=p)
ov = overlap_diagnostics(p)
print("\nbalance |SMD| max: before=%.3f -> after=%.3f" % (bal["max_abs_smd_before"], bal["max_abs_smd_after"]))
print("overlap: frac in [0.1,0.9]=%.3f, min=%.3f, max=%.3f" % (ov["frac_in_0.1_0.9"], ov["min"], ov["max"]))

print("\n=== Сводная оценка качества adjustment set ===")
q = evaluate_adjustment_set_quality(reduced, t, y, ground_truth_effect=TRUE_ATE)
display(pd.Series(q).round(4))

=== Оценки ATE по методам ===

balance |SMD| max: before=0.869 -> after=0.019
overlap: frac in [0.1,0.9]=0.977, min=0.018, max=0.974

=== Сводная оценка качества adjustment set ===


,ATE,bias_vs_true
naive,2.145,-0.855
IPW (propensity),2.930,-0.070
doubly_robust,2.948,-0.052


ate_naive                      2.1449
ate_ipw                        2.9299
ate_dr                         2.9484
max_abs_smd_before             0.8694
max_abs_smd_after              0.0188
overlap_frac_in_0.1_0.9        0.9765
ground_truth                   3.0000
bias_naive                    -0.8551
bias_ipw                      -0.0701
bias_dr                       -0.0516
abs_bias_reduction_ipw_pct    91.7969
abs_bias_reduction_dr_pct     93.9655
dtype: float64

## 3. Выводы (single-table)

- Если эмбеддинги покрывают конфаундеры, IPW/doubly-robust уменьшают смещение наивной оценки ATE, а баланс |SMD| падает после поправки.
- Overlap-диагностика проверяет применимость взвешивания: при propensity у краёв [0,1] оценки нестабильны.
- Production-адаптеры на pyspark (`EmbeddingReducer`, `PropensityScorer`) и сравнение по многим сценариям — в исследовательском `notebook.ipynb`; здесь фокус на одной таблице.